# SPATIAL INTELLIGENCE - PART 1

In [ ]:
# This cell is not needed if you have pip installed topologicpy
# import sys
# sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Import the main House OBJ file

In [ ]:
House = Topology.ByOBJPath(r"D:\3rd sem\GRAPH MACHINE LEARNING\ASSIGNMENTS\1\complex house.obj", selfMerge=False)
print("House is a list")
print(House)

## 5. Convert List to Cluster


In [ ]:
# Convert House list to a Cluster first
house_cluster = Cluster.ByTopologies(House)
house = CellComplex.ByFacesCluster(house_cluster)
print("House is a cell complex")
print(len(House))


## 6. Import Doors and Windows OBJ file

In [ ]:
door = Topology.ByOBJPath(r"D:\3rd sem\GRAPH MACHINE LEARNING\ASSIGNMENTS\1\complex_door.obj", selfMerge=False)
door_c = Cluster.ByTopologies(door)
door = Topology.Faces(door_c)
print(f"Number of doors in cellcomplex: {len(door) if door else 0}")



window = Topology.ByOBJPath(r"D:\3rd sem\GRAPH MACHINE LEARNING\ASSIGNMENTS\1\complex_window.obj", selfMerge=False)
window_c = Cluster.ByTopologies(window)
window = Topology.Faces(window_c)
print(f"Number of windows in cellcomplex: {len(window) if window else 0}")

## 7. Cell Extraction and Color Tagging

In [ ]:
cells = []
selectors = []

# Define color map using RGB tuples (0-1 range)
color_map = {
    "Living_Room": "red",
    "Kitchen": "yellow",
    "Dining_Room": "pink",
    "Bedroom": "blue",
    "Bathroom": "purple",
    "Exterior_Corridor": "grey",
    "Interior_Corridor": "cyan",
    "Door": "brown",
    "Window": "light cyan"
}

# Extract cells from ORIGINAL House geometry (before apertures) to get room names
print(f"Processing {len(House)} original geometries...\n")

for obj in House:
    # Get the dictionary from the original object
    d = Topology.Dictionary(obj)
    name = Dictionary.ValueAtKey(d, "name") if d else None
    
    # Get color from color_map (default to gray if not found)
    color = color_map.get(name, [0.83, 0.83, 0.83])
    
    print(f"Room: {name} → color: {color}")
    
    # Extract cells from this object
    merged = Topology.SelfMerge(obj)
    cells_from_obj = Topology.Cells(merged) or []
    
    # Tag each cell with room name and color
    for cell in cells_from_obj:
        d_tagged = Dictionary.ByKeysValues(["name", "color"], [name, color])
        c = Topology.SetDictionary(cell, d_tagged)
        cells.append(c)
        selector = Topology.InternalVertex(cell)
        s = Topology.SetDictionary(selector, d_tagged)
        selectors.append(s)

# Add doors with their color
print(f"\nProcessing {len(door) if door else 0} doors...")
for door_face in (door or []):
    d_door = Dictionary.ByKeysValues(["name", "color"], ["Door", "brown"])
    door_tagged = Topology.SetDictionary(door_face, d_door)
    cells.append(door_tagged)

# Add windows with their color
print(f"Processing {len(window) if window else 0} windows...")
for window_face in (window or []):
    d_window = Dictionary.ByKeysValues(["name", "color"], ["Window", "light cyan"])
    window_tagged = Topology.SetDictionary(window_face, d_window)
    cells.append(window_tagged)

print(f"\nTotal cells extracted: {len(cells)} (rooms + doors + windows)")
print(len(cells))

In [ ]:
Topology.Show(cells,
               faceColorKey="color", backgroundColor="white", renderer=renderer)

## 8. Creating CellComplex

In [ ]:
cc= Topology.AddApertures(house, door, exclusive=False, subTopologyType="face")
cc= Topology.AddApertures(house, window, exclusive=False, subTopologyType="face")  

## 9. Aperture Data Extraction & Connectivity Graph Creation

In [ ]:
# door and window are already Face objects from OBJ files
aperture_faces = (door or []) + (window or [])

print(f"Found {len(aperture_faces)} aperture faces")
print(f"  Doors: {len(door) if door else 0}")
print(f"  Windows: {len(window) if window else 0}")

In [ ]:
aperture_data = []

for face in aperture_faces:
    f_centroid = Topology.Centroid(face)
    d = Topology.Dictionary(face)
    
    f_color = Dictionary.ValueAtKey(d, "color") if d else "brown"  # default
    f_type = Dictionary.ValueAtKey(d, "name") if d else "Aperture"
    
    aperture_data.append((f_centroid, f_color, f_type))

In [ ]:
g = Graph.ByTopology(
    cc,
    direct=False,
    viaSharedApertures=True,
    toExteriorApertures=True
)
print("Graph vertices:", len(Graph.Vertices(g)))
print("Edges:", len(Graph.Edges(g)))

In [ ]:
import math

# Get color from color_map (default to gray if not found)
color = color_map.get(name, [0.83, 0.83, 0.83])

verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))
edges = Graph.Edges(g)


for e in edges:
    d = Topology.Dictionary(e)
    print(Dictionary.Keys(d), Dictionary.Values(d)) 


# Step 1: Precompute cell centroids, areas, and colors
cell_data = []

# Color name to hex mapping
color_hex_map = {
    "red": "#FF0000",
    "blue": "#0000FF",
    "yellow": "#FFFF00",
    "pink": "#FF1493",
    "purple": "#800080",
    "cyan": "#00FFFF",
    "grey": "#808080",
    "brown": "#8B4513",
    "light cyan": "#E0FFFF"
}

for cell in cells:
    c_centroid = Topology.Centroid(cell)
    c_area = Cell.SurfaceArea(cell)
    d = Topology.Dictionary(cell)
    c_color = Dictionary.ValueAtKey(d, "color") if d else "grey"
    # Convert to hex if it's a color name
    c_color_hex = color_hex_map.get(c_color, "#808080")
    cell_data.append((c_centroid, c_area, c_color_hex))

# Step 2: min/max
areas = [a for _, a, _ in cell_data]
min_area = min(areas)
max_area = max(areas)

# Step 3: assign size and color to room vertices
updated_verts = []

for i, v in enumerate(verts):
    v_centroid = Topology.Centroid(v)

    # Find closest cell
    min_dist = float("inf")
    closest_area = min_area  # fallback
    closest_color = "grey"  # default color

    for c_centroid, c_area, c_color in cell_data:
        d = Vertex.Distance(v_centroid, c_centroid)
        if d < min_dist:
            min_dist = d
            closest_area = c_area
            closest_color = c_color

    # Proper normalization
    norm = (closest_area - min_area) / (max_area - min_area + 1e-6)

    # Better visual scaling
    vertex_size = 8 + 20* (norm ** 0.5)  # adjust exponent for better spread

    # Assign dictionary with size and color
    d = Topology.Dictionary(v)
    if d is None:
        d = Dictionary.ByKeysValues([], [])

    d = Dictionary.SetValueAtKey(d, "size", vertex_size)
    d = Dictionary.SetValueAtKey(d, "color", closest_color)
    v_updated = Topology.SetDictionary(v, d)
    updated_verts.append(v_updated)

    if i < 5:
        print(f"Vertex {i}: Area={closest_area:.2f}, Size={vertex_size:.2f}, Color={closest_color}")



In [ ]:
# Step 4: Create vertices for doors with brown color
aperture_verts = []
aperture_edges = []

# Color name to hex mapping
color_hex_map = {
    "red": "#FF0000",
    "blue": "#0000FF",
    "yellow": "#FFFF00",
    "pink": "#FF1493",
    "purple": "#800080",
    "cyan": "#00FFFF",
    "grey": "#808080",
    "brown": "#8B4513",
    "light cyan": "#E0FFFF"
}

for door_face in (door or []):
    door_centroid = Topology.Centroid(door_face)
    door_vertex = Vertex.ByCoordinates(door_centroid.X, door_centroid.Y, door_centroid.Z)
    
    if door_vertex:
        # Set dictionary for door vertex with hex color
        d_door = Dictionary.ByKeysValues(["size", "color"], [5, "#8B4513"])  # brown hex
        door_vertex = Topology.SetDictionary(door_vertex, d_door)
        aperture_verts.append(door_vertex)

# Step 5: Create vertices for windows with light cyan color
for window_face in (window or []):
    window_centroid = Topology.Centroid(window_face)
    window_vertex = Vertex.ByCoordinates(window_centroid.X, window_centroid.Y, window_centroid.Z)
    
    if window_vertex:
        # Set dictionary for window vertex with hex color
        d_window = Dictionary.ByKeysValues(["size", "color"], [4, "#E0FFFF"])  # light cyan hex
        window_vertex = Topology.SetDictionary(window_vertex, d_window)
        aperture_verts.append(window_vertex)

print(f"Created {len(aperture_verts)} aperture vertices (doors + windows)")

# Step 6: Combine all vertices
all_verts = updated_verts + aperture_verts
print(f"Total vertices: {len(all_verts)} (rooms + apertures)")

# Step 7: rebuild graph with all vertices and edges
g = Graph.ByVerticesEdges(all_verts, Graph.Edges(g))

print(f" Updated graph with {len(all_verts)} vertices (including {len(aperture_verts)} aperture vertices)")


for e in Graph.Edges(g):
    d = Dictionary.ByKeysValues(["width", "color"], [10, "black"])
    e = Topology.SetDictionary(e, d) 


## 10.Visualisation

In [ ]:
Topology.Show(g, cc, door, window, 
               vertexSizeKey="size", vertexColorKey="color", backgroundColor="white", renderer=renderer)